# Setup

In [1]:
# Standard library imports
import gc
import warnings

# Third-party library imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

# Machine Learning imports
import lightgbm as lgb
from sklearn.metrics import mean_squared_error as mse
from sklearn.model_selection import (
    GroupKFold,
    KFold, 
    TimeSeriesSplit, 
    train_test_split,
)

# Configuration
warnings.simplefilter("ignore", FutureWarning)

Right this way: https://open.substack.com/pub/konradb/p/this-week-in-tech-2032026

TODO

In [8]:
# general settings
class CFG:
    data_folder = './data/'
    graph_folder = './graphs/'
    img_dim1 = 20
    img_dim2 = 10
    SEED = 42
    metric = 'rmse'


# display style 
plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (CFG.img_dim1, CFG.img_dim2)

np.random.seed(CFG.SEED)

TODO

# Utils

In [9]:
def train_and_predict(
    X_train, y_train,
    X_val, y_val, X_test=None,
    params=None, early_stopping_rounds=100,
):
    model = lgb.LGBMRegressor(**params)

    model.fit(
        X_train, y_train, eval_set=[(X_val, y_val)],
        eval_metric="rmse",
        callbacks=[  lgb.early_stopping(early_stopping_rounds),  lgb.log_evaluation(50),
        ],)

    val_pred = model.predict(X_val)

    test_pred = None
    if X_test is not None:
        test_pred = model.predict(X_test)

    return model, val_pred, test_pred

TODO

# Data preparation

Taken from: https://www.kaggle.com/datasets/robikscube/ubiquant-parquet

In [11]:
xtrain = pd.read_parquet(f"{CFG.data_folder}train_low_mem.parquet")
xtrain.head(10)

,row_id,time_id,investment_id,target,f_0,f_1,f_2,f_3,f_4,f_5,...,f_290,f_291,f_292,f_293,f_294,f_295,f_296,f_297,f_298,f_299
0,0_1,0,1,-0.300875,0.932573,0.113691,-0.402206,0.378386,-0.203938,-0.413469,...,0.366028,-1.095620,0.200075,0.819155,0.941183,-0.086764,-1.087009,-1.044826,-0.287605,0.321566
1,0_2,0,2,-0.231040,0.810802,-0.514115,0.742368,-0.616673,-0.194255,1.771210,...,-0.154193,0.912726,-0.734579,0.819155,0.941183,-0.387617,-1.087009,-0.929529,-0.974060,-0.343624
2,0_6,0,6,0.568807,0.393974,0.615937,0.567806,-0.607963,0.068883,-1.083155,...,-0.138020,0.912726,-0.551904,-1.220772,-1.060166,-0.219097,-1.087009,-0.612428,-0.113944,0.243608
3,0_7,0,7,-1.064780,-2.343535,-0.011870,1.874606,-0.606346,-0.586827,-0.815737,...,0.382201,0.912726,-0.266359,-1.220772,0.941183,-0.609113,0.104928,-0.783423,1.151730,-0.773309
4,0_8,0,8,-0.531940,0.842057,-0.262993,2.330030,-0.583422,-0.618392,-0.742814,...,-0.170365,0.912726,-0.741355,-1.220772,0.941183,-0.588445,0.104928,0.753279,1.345611,-0.737624
5,0_9,0,9,1.505904,0.608855,1.369305,-0.761515,0.865860,-0.359269,-1.835762,...,0.333684,-1.095620,-0.335999,0.819155,-1.060166,-0.343812,-1.087009,0.077862,0.142943,-0.055550
6,0_10,0,10,-0.260731,-1.863797,0.113691,1.573864,-0.598433,-0.569936,0.398784,...,0.821560,0.912726,0.476309,-1.220772,0.941183,-0.434315,1.296864,0.171329,1.051288,-0.745335
7,0_12,0,12,-0.469207,0.408954,-0.765238,0.261430,-0.591895,-0.037260,0.668721,...,0.821560,-1.095620,-0.864354,-1.220772,-1.060166,-0.300218,1.296864,-0.779556,0.274961,-0.182520
8,0_13,0,13,0.094525,0.861187,2.373796,-1.148977,0.752205,-0.050502,-2.249047,...,-0.658241,0.912726,0.718282,0.819155,0.941183,4.198117,1.296864,1.854434,0.000000,-0.688340
9,0_14,0,14,-0.251120,-2.476555,0.239253,2.222353,-0.582276,-0.618236,0.386263,...,0.821560,-1.095620,-0.615709,-1.220772,-1.060166,-0.647769,0.104928,-0.849789,0.805876,-0.820165
